Installing Unsloth

In [11]:
%%capture
!pip install unsloth
# !pip install transformers>=4.54.1

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [12]:
%%capture
!pip install --no-deps git+https://github.com/huggingface/transformers.git 
!pip install --no-deps --upgrade timm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [4]:
from unsloth import FastModel
import torch
from IPython.display import Image, display
import gc
import json

torch._dynamo.config.cache_size_limit = 256

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-07-27 06:38:20.045430: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753598300.072801     280 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753598300.080998     280 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


Downloading FastModel-Unsloth

In [5]:
fourbit_models = [
    "unsloth/gemma-3n-E4B-it-unsloth-bnb-4bit",
    "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit",
    "unsloth/gemma-3n-E4B-unsloth-bnb-4bit",
    "unsloth/gemma-3n-E2B-unsloth-bnb-4bit",
]
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3n-E4B-it",
    dtype = None,
    max_seq_length = 4096,
    load_in_4bit = True,
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = True, 
    finetune_language_layers   = True,  
    finetune_attention_modules = True,  
    finetune_mlp_modules       = True,  

    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

==((====))==  Unsloth 2025.7.8: Fast Gemma3N patching. Transformers: 4.55.0.dev0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3N does not support SDPA - switching to eager!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.72G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [6]:
def do_gemma_3n_inference(model, tokenizer, messages, max_new_tokens = 4096):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda:0")

    generated_token_ids = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64,
    )

    
    generated_text = tokenizer.decode(generated_token_ids[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)

    del inputs
    del generated_token_ids 
    torch.cuda.empty_cache()
    gc.collect()

    return generated_text

Showing image we'll be working on

In [7]:
report_image = "https://raw.githubusercontent.com/DeepLumiere/MedOCR/master/beta/img.png"

# Display the image directly from the URL
display(Image(url=report_image))

Testing Gemma-3n with image + text

In [8]:
report_image = "https://raw.githubusercontent.com/DeepLumiere/MedOCR/master/beta/img.png"

list_tests_messages = [{
    "role": "user",
    "content": [
        { "type": "image", "image": report_image },
        { "type": "text", "text": "From this medical report image, please list all the distinct medical test names you can identify. Present them as a comma-separated list." }
    ]
}]

print("--- Extracting Test Features from the Report Image ---")
print("Please wait, the model is analyzing the image...")

raw_test_list_output = do_gemma_3n_inference(model, tokenizer, list_tests_messages, max_new_tokens=256)

test_names = []
if ',' in raw_test_list_output:
    test_names = [name.strip() for name in raw_test_list_output.split(',') if name.strip()]
else:
    for line in raw_test_list_output.split('\n'):
        clean_line = line.strip()
        if clean_line and (clean_line.startswith(('* ', '- ', '• ')) or (clean_line[0].isdigit() and '. ' in clean_line[:3])):
            clean_line = clean_line.split('. ', 1)[-1].strip() 
        if clean_line:
            test_names.append(clean_line)

print("\n------- Identified Test Features (Copy/Type one into the next cell) -------")
items_per_row = 3
for i, test_name in enumerate(test_names):
    print(f"{test_name:<30}", end="") 
    if (i + 1) % items_per_row == 0:
        print() 
print()
print("------------------------------------------------------------------------------")

--- Extracting Test Features from the Report Image ---
Please wait, the model is analyzing the image...

--- Identified Test Features (Copy/Type one into the next cell) ---
Hemoglobin                    Total Leukocyte Count         Differential Leukocyte Count  
Neutrophils                   Lymphocyte                    Eosinophils                   
Monocytes                     Basophils                     Platelet Count                
Total RBC Count               Hematocrit Value              HCT                           
Mean Corpuscular Volume       MCV                           Mean Cell Hemoglobin          
MCH                           Mean Cell Hemoglobin ConcentrationMCHC.                         

---------------------------------------------------------------


In [9]:
finder = "MCHC" #input("Enter the test name you want to review: ")

messages = [{
    "role" : "user",
    "content": [
        { "type": "image", "image" : report_image },
        { "type": "text",  "text" : finder + " . Find value, review whether it's in normal range, if not give potential reasons." }
    ]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda:0")
outputs = model.generate(
    **inputs,
    max_new_tokens = 2048,
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

labreport = do_gemma_3n_inference(model, tokenizer, messages, max_new_tokens = 128)

Enter the test name you want to review:  MCHC


Finetuned Model

In [10]:
print(labreport)

**MCHC Value:** 35.7

**Normal Range:** 31.5 - 34.5

**Review:** The MCHC value of 35.7 is **within the normal range**.

**Therefore, there is no need to provide potential reasons for it being out of range.**


In [ ]:
model.save_pretrained("gemma-3n") 
tokenizer.save_pretrained("gemma-3n")
